In [42]:
import re
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack, save_npz
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer #(count one for LDA strictly)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import LatentDirichletAllocation

In [43]:
import os
import joblib

In [23]:
import nltk
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/raajjyoti/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [24]:
r_stoplist = set(nltk.corpus.stopwords.words("english"))
print(r_stoplist)

{'all', 'having', "that'll", 'can', 'she', 'will', 'a', 'over', "they've", "he'd", 'which', 'after', 'up', 'were', "isn't", "mustn't", 'as', 'our', "it'll", "shan't", "couldn't", 'very', 'on', "he's", 'own', 'same', 'again', 'here', "weren't", "you'll", 'their', "we'd", "wouldn't", 'yourself', 'yourselves', 'are', 'its', 'hasn', 'or', 'them', 'herself', 'down', "he'll", 'm', 't', 'being', 'but', 'your', 'my', 'each', 'out', 'under', 'needn', 'have', "mightn't", "it's", 'themselves', 'about', 'weren', 'does', "aren't", 'doing', 'both', 'not', 'now', 'off', 'ours', "didn't", 'then', "we've", 'is', 'before', 'had', "they're", 'only', 'for', 'from', 'by', 'did', 'above', 'll', "i'd", 'they', "shouldn't", 'd', 'him', 'we', "we're", 'against', 'to', 'the', "haven't", 'too', 'most', 'when', "needn't", 'than', "should've", 'doesn', 'while', 'those', 'with', 'so', 'isn', 'any', 'do', 'some', 'of', 'don', 'there', 'because', 'these', "she'd", 'it', "hadn't", 'couldn', 'once', "she'll", 'through'

In [25]:
news_sw = ["said", "says", "told", "according", "reported", "reuters", "getty",
    "images", "image", "photo", "video", "watch", "read", "featured",
    "via", "twitter", "facebook", "post", "article", "news", "year",
    "years", "time", "day", "today", "new"]

stoplist = {word.replace("'","") for word in r_stoplist}.union(news_sw)
stoplist.discard("ill")

print(stoplist) 

{'wont', 'all', 'having', 'shed', 'im', 'can', 'she', 'will', 'a', 'over', 'which', 'after', 'up', 'were', 'according', 'article', 'watch', 'as', 'our', 'hed', 'very', 'photo', 'on', 'own', 'reported', 'same', 'again', 'here', 'doesnt', 'mustnt', 'their', 'video', 'yourself', 'via', 'facebook', 'yourselves', 'are', 'dont', 'its', 'hasn', 'werent', 'or', 'them', 'theyve', 'herself', 'mightnt', 'says', 'featured', 'down', 'm', 'shouldve', 't', 'neednt', 'being', 'but', 'your', 'my', 'each', 'out', 'under', 'youve', 'news', 'needn', 'have', 'well', 'themselves', 'about', 'weren', 'wouldnt', 'does', 'doing', 'ours', 'both', 'not', 'now', 'off', 'shant', 'then', 'wed', 'theyll', 'is', 'before', 'had', 'havent', 'only', 'id', 'for', 'from', 'by', 'did', 'above', 'll', 'they', 'd', 'him', 'we', 'youd', 'against', 'to', 'the', 'told', 'too', 'image', 'most', 'couldnt', 'when', 'images', 'than', 'with', 'doesn', 'while', 'those', 'so', 'isn', 'isnt', 'any', 'do', 'some', 'of', 'don', 'itd', 'th

In [26]:
extra_stopwords = {
    # Attribution verbs
    "said", "says", "say", "saying", "told", "according", "reported", "reports",
    "reporting", "stated", "claimed", "claims", "asked", "added", "explained",
    "wrote", "noted", "announced", "confirmed",
    # Media & scraping boilerplate
    "reuters", "getty", "images", "image", "photo", "photos", "video", "videos",
    "watch", "read", "featured", "via", "source", "copyright", "subscribe",
    "click", "follow", "share", "comment", "comments", "twitter", "facebook",
    "youtube", "instagram", "embed", "pic", "post", "posts", "article", "news",
    # Dataset publisher tags
    "21wire", "21stcenturywire", "breitbart", "infowars", "dailycaller", "huffpost", "buzzfeed",
    # Calendar & generic discourse filler
    "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday",
    "january", "february", "march", "april", "may", "june", "july", "august",
    "september", "october", "november", "december",
    "year", "years", "month", "months", "week", "weeks", "day", "days",
    "time", "times", "today", "yesterday", "ago", "recent", "recently",
    "would", "could", "also", "one", "two", "three", "like", "get", "got",
    "make", "made", "going", "way", "even", "still", "back", "new", "first",
    "last", "many", "much", "well", "use", "used", "using", "want", "know",
    "think", "see", "look", "yearsold",
}

final_stoplist = sorted(stoplist.union(extra_stopwords))

In [38]:
file_clean = "WELFake2.csv"
#PUBLISHER_LEAK_PAT = re.compile(r"\b(?:reuters|21wire|21stcenturywire|breitbart|infowars)\b", flags=re.I)

df_clean = pd.read_csv(file_clean)

In [28]:
df_clean.columns

Index(['label', 'title_len', 'text_len', 'excl_count', 'quest_count',
       'caps_words', 'text3', 'title3', 'clean_news'],
      dtype='str')

In [29]:
#df_clean["clean_news"] = df_clean["clean_news"].str.replace(PUBLISHER_LEAK_PAT, " ", regex=True)

In [30]:
num_cols = ["title_len", "text_len", "excl_count", "quest_count", "caps_words"]
num_features = df_clean[num_cols] 

In [31]:
#first dimensions of all input arrays must be same or else it wont split
#here the input arrays are df_clean["clean_news"], num_features, df_clean["label"]
X_train_txt, X_test_txt, X_train_num, X_test_num, y_train, y_test = train_test_split(
    df_clean["clean_news"],
    num_features,
    df_clean["label"].astype(int),
    test_size=0.2, #20% of data used for testing, rest go to train 
    stratify=df_clean["label"],
    random_state=42,
)

In [39]:
lda_vect = CountVectorizer(
    stop_words=final_stoplist,
    max_df=0.85,
    min_df=20,
    token_pattern=r"\b[a-z0-9]{3,}\b",
)

In [40]:
X_train_counts = lda_vect.fit_transform(X_train_txt)
X_test_counts = lda_vect.transform(X_test_txt)

In [41]:
#12 topic LDA decomposition for algorithms that are non-parametric estimators
lda_model = LatentDirichletAllocation(
    n_components=12,           #num_topics
    learning_method="online", #batch-wise stochastic (EM) update per epoch 
    batch_size=2048, 
    max_iter=15, #num_epochs
    random_state=42,
    n_jobs=-1,
)

X_train_topics = lda_model.fit_transform(X_train_counts)
X_test_topics = lda_model.transform(X_test_counts)

In [45]:
X_train_topics.shape, X_test_topics.shape

((57656, 12), (14414, 12))

#Term Frequency-Inverse Document Frequency (TF-IDF)
- Removes disadvantage of the Bag of Words method 
- create matrix of tfidf (term-frequency inverse document frquency) vectors
- These are basically weights assigned to each word based on occurence in the doc and across whole dataset 
- IDF of rare term is likely higher than IDF of frequent term but that depends on setting on min_df and max_df 

In [33]:
w_matrix = TfidfVectorizer(
        ngram_range=(1,2), #unigram(bow) + bigrams
        min_df=10, #dont know why they raise warning on giving float inputs here 
        max_df=0.90, #prunes words occuring in more than 90% of documents (noise) 
        sublinear_tf=True,
        max_features=100_000,
        token_pattern=r"\b[a-z0-9]{2,}\b",
        )

X_train_tfidf = w_matrix.fit_transform(X_train_txt)
X_test_tfidf = w_matrix.transform(X_test_txt) 

In [34]:
#X_train_tfidf.shape, X_test_tfidf.shape

In [35]:
#Scale stylometric and numerical features into sparse matrices 
scaler = StandardScaler()
X_train_num_scaled = csr_matrix(scaler.fit_transform(X_train_num))
X_test_num_scaled = csr_matrix(scaler.transform(X_test_num)) #transform only since it fits data based on the mean and std of the previous X_train_num_scaled 

In [36]:
X_train_final = hstack([X_train_tfidf, X_train_num_scaled], format="csr")
X_test_final = hstack([X_test_tfidf, X_test_num_scaled], format="csr")

In [37]:
X_train_final.shape, X_test_final.shape

((57656, 100005), (14414, 100005))

In [46]:
os.makedirs("norm_data", exist_ok=True)

# 1. Shared target labels (dense 1D arrays)
np.save("norm_data/y_train.npy", np.asarray(y_train))
np.save("norm_data/y_test.npy", np.asarray(y_test))

# 2. Case 1: Stylometric / Numerical features (57656 x 5)
save_npz("norm_data/X_train_num.npz", X_train_num_scaled)
save_npz("norm_data/X_test_num.npz", X_test_num_scaled)

# 3. Case 2: TF-IDF N-gram features (57656 x 100000)
save_npz("norm_data/X_train_tfidf.npz", X_train_tfidf)
save_npz("norm_data/X_test_tfidf.npz", X_test_tfidf)

# 4. Case 3: LDA Topic distributions (57656 x 12)
np.save("norm_data/X_train_topics.npy", X_train_topics)
np.save("norm_data/X_test_topics.npy", X_test_topics)

# 5. Case 4: Combined TF-IDF + Stylometrics (57656 x 100005)
save_npz("norm_data/X_train_final.npz", X_train_final)
save_npz("norm_data/X_test_final.npz", X_test_final)

# 6. Fitted preprocessors (for feature-weight inspection later)
joblib.dump(w_matrix, "norm_data/tfidf_vectorizer.joblib")
joblib.dump(scaler, "norm_data/num_scaler.joblib")
joblib.dump(lda_model, "norm_data/lda_model.joblib")

['norm_data/lda_model.joblib']